# Generate Datasets for Classification Training of HydraBLE

In [1]:
import os
from pathlib import Path


os.chdir(Path.cwd().parents[0])
print("Now in:", Path.cwd())

dataPathProcessed = str(Path.cwd()) + r"\data\csv" + r"\Processed Files\\"

Now in: C:\Users\stsax\OneDrive\Studium\9. Semester\Masterarbeit\Repository


In [2]:
SEQUENCE_LENGTH = 64
STRIDE = 1
UNKNOWN_RATIO = 0.5

In [3]:
import math
from data_processing.preprocessing_functions import prepare_real_dataset, prepare_synthetic_dataset, \
    create_stream_idx_df
import pandas as pd
from data_processing.LabelLut import LABEL_OTHER_DEVICE, LABEL_GOOGLE_FIND_MY, STATE_LOST, SEPARATOR, STATE_NEARBY, \
    LABEL_TILE_NETWORK

datasets = {r"labeled_and_aggregated_data": ('real','mixed'),
            r"synthetic_samples_other_device": ("synthetic", "unknown"),
            "synthetic_samples_adversarial_google_find_my": ("synthetic", "unknown"),
            "synthetic_samples_adversarial_samsung_smart_things": ("synthetic", "unknown"),
            "synthetic_samples_adversarial_tile": ("synthetic", "unknown"),
            "synthetic_samples_dult_nearby": ("synthetic", "known"),
            "synthetic_samples_dult_lost": ("synthetic", "known"),
            }


SIZES_KNOWN_CLASS =  {'train': 1_000, 'validation': 200, 'test': 200}
STORED_SEQUENCES_SYNTHETIC = {'train': 1, 'validation': 1, 'test': 1}

REPLACE_LABEL_LUT = {LABEL_GOOGLE_FIND_MY + SEPARATOR + STATE_NEARBY: LABEL_GOOGLE_FIND_MY + SEPARATOR + STATE_LOST,
                     LABEL_TILE_NETWORK + SEPARATOR + STATE_NEARBY: LABEL_TILE_NETWORK + SEPARATOR + STATE_LOST
                     }



for suffix in ['train', 'validation', 'test']:
    SIZE_KNOWN_CLASS_FOR_SUFFIX = SIZES_KNOWN_CLASS[suffix]
    STORED_SEQUENCES_SYNTHETIC_FOR_SUFFIX = STORED_SEQUENCES_SYNTHETIC[suffix]

    sets = []

    for path, _ in datasets.items():
        dataset = pd.read_parquet(dataPathProcessed + path + "_" + suffix + ".parquet")

        sets.append(dataset[['Label']])

    dataset = pd.concat(sets)
    KNOWN_CLASSES = sorted(list(dataset['Label'].unique()))

    if LABEL_OTHER_DEVICE in KNOWN_CLASSES:
        KNOWN_CLASSES.remove(LABEL_OTHER_DEVICE)

    NUM_OF_KNOWN_CLASSES = len(KNOWN_CLASSES)


    datasets_known = []
    datasets_synthetic = []

    for idx, (path, (dataset_type, sample_type)) in enumerate(datasets.items()):
        dataset = pd.read_parquet(dataPathProcessed + path + "_" + suffix + ".parquet")

        if dataset_type == "real":
            dataset = prepare_real_dataset(dataset)
            dataset = dataset[dataset['Label'] != LABEL_OTHER_DEVICE]

            datasets_known.append(dataset)


        elif dataset_type == "synthetic":
            dataset = dataset[['Time', 'Source', 'Channel', 'RSSI', 'Hex Data', 'Label']]

            if sample_type == 'unknown':
                datasets_synthetic.append(dataset)

            elif sample_type == 'known':
                dataset = prepare_synthetic_dataset(dataset, n_sample=math.ceil(SIZE_KNOWN_CLASS_FOR_SUFFIX / STORED_SEQUENCES_SYNTHETIC_FOR_SUFFIX), n_repeat=SEQUENCE_LENGTH + (STORED_SEQUENCES_SYNTHETIC_FOR_SUFFIX - 1) * STRIDE, file_id=idx)

                datasets_known.append(dataset)



    datasets_for_suffix = []

    for dataset in datasets_known:
        datasets_for_suffix.append(dataset)


    dataset_synthetic = prepare_synthetic_dataset(pd.concat(datasets_synthetic), n_sample=math.ceil(NUM_OF_KNOWN_CLASSES * SIZE_KNOWN_CLASS_FOR_SUFFIX / STORED_SEQUENCES_SYNTHETIC_FOR_SUFFIX),  n_repeat=SEQUENCE_LENGTH + (STORED_SEQUENCES_SYNTHETIC_FOR_SUFFIX - 1) * STRIDE, file_id=1)

    datasets_for_suffix.append(dataset_synthetic)
    dataset_for_suffix = pd.concat(datasets_for_suffix)


    dataset_for_suffix['Label'] = dataset_for_suffix['Label'].replace(REPLACE_LABEL_LUT)

    classes = list(dataset_for_suffix['Label'].unique())
    classes.remove(LABEL_OTHER_DEVICE)

    LABEL_TARGETS = {c: SIZE_KNOWN_CLASS_FOR_SUFFIX for c in classes}
    LABEL_TARGETS[LABEL_OTHER_DEVICE] = int(UNKNOWN_RATIO * sum(LABEL_TARGETS.values()) / (1 - UNKNOWN_RATIO))


    dataset, sequence_table, stream_index = create_stream_idx_df(dataset_for_suffix, label_targets=LABEL_TARGETS,
                                                                 label_column='Label', time_column='Time',
                                                                 stride=STRIDE, source_column='Source',
                                                                 file_column='File', sequence_length=SEQUENCE_LENGTH)


    dataset['Time Delta'] = dataset['Time Delta'].apply(lambda x: min(x, 2**32 - 1))
    dataset['Time Delta'] = dataset['Time Delta'].apply(lambda x: max(x, 0))

    dataset.to_parquet(dataPathProcessed + "classification_dataset_v2_" + suffix + ".parquet")
    sequence_table.to_parquet(dataPathProcessed + "classification_sequence_table_v2_" + suffix + ".parquet")
    stream_index.to_parquet(dataPathProcessed + "classification_stream_index_v2_" + suffix + ".parquet")


In [4]:
stream_index

,Stream_ID,Sequence_ID,Label,Start_Packet_IDX,Length
0,0,0,Apple Device,1198,64
1,1,0,Apple Device,1493,64
2,2,0,Apple Device,1993,64
3,3,1,Apple Device,237,64
4,4,1,Apple Device,589,64
...,...,...,...,...,...
5995,5995,3624,Other Device,0,64
5996,5996,3625,Other Device,0,64
5997,5997,3626,Other Device,0,64
5998,5998,3627,Other Device,0,64


In [5]:
dataset

,Time,Time Delta,Source,Channel,RSSI,Hex Data,Label,File,Sequence_ID,Packet_in_Sequence
0,1.712647e+09,0,5926065A4414,37,33,D6BE898E462514445A0626591EFF4C00071901132020F2...,Apple Device,Apple iDevices\Apple AirPod\Apple_AirPod_(lost),0,0
1,1.712647e+09,878,5926065A4414,38,33,D6BE898E462514445A0626591EFF4C00071901132020F2...,Apple Device,Apple iDevices\Apple AirPod\Apple_AirPod_(lost),0,1
2,1.712647e+09,876,5926065A4414,39,33,D6BE898E462514445A0626591EFF4C00071901132020F2...,Apple Device,Apple iDevices\Apple AirPod\Apple_AirPod_(lost),0,2
3,1.712647e+09,184483,5926065A4414,37,33,D6BE898E462514445A0626591EFF4C00071901132020F2...,Apple Device,Apple iDevices\Apple AirPod\Apple_AirPod_(lost),0,3
4,1.712647e+09,878,5926065A4414,38,33,D6BE898E462514445A0626591EFF4C00071901132020F2...,Apple Device,Apple iDevices\Apple AirPod\Apple_AirPod_(lost),0,4
...,...,...,...,...,...,...,...,...,...,...
1154902,1.771674e+09,792,DC90654758FA,38,33,D6BE898E601BFA58476590DC0201060303EDFE0D16EDFE...,Tile Tracker (lost),BLE Tracker\Tile\Tile Slim\Tile_Slim_(nearby),4010,6344
1154903,1.771674e+09,793,DC90654758FA,39,33,D6BE898E601BFA58476590DC0201060303EDFE0D16EDFE...,Tile Tracker (lost),BLE Tracker\Tile\Tile Slim\Tile_Slim_(nearby),4010,6345
1154904,1.771674e+09,1999212,DC90654758FA,37,33,D6BE898E601BFA58476590DC0201060303EDFE0D16EDFE...,Tile Tracker (lost),BLE Tracker\Tile\Tile Slim\Tile_Slim_(nearby),4010,6346
1154905,1.771674e+09,793,DC90654758FA,38,32,D6BE898E601BFA58476590DC0201060303EDFE0D16EDFE...,Tile Tracker (lost),BLE Tracker\Tile\Tile Slim\Tile_Slim_(nearby),4010,6347


In [6]:
sequence_table

,Sequence_ID,Label,File,Source,Start_Row,Num_of_Packets,Num_of_Streams
0,0,Apple Device,Apple iDevices\Apple AirPod\Apple_AirPod_(lost),5926065A4414,0,2408,2345
1,1,Apple Device,Apple iDevices\Apple AirPod\Apple_AirPod_(lost),6B86F30DADDE,2408,14442,14379
2,2,Apple Device,Apple iDevices\Apple AirPod\Apple_AirPod_(lost),6FB0DD44473A,16850,14451,14388
3,3,Apple Device,Apple iDevices\Apple AirPod\Apple_AirPod_(lost),72D872163974,31301,9108,9045
4,4,Apple Device,Apple iDevices\Apple AirPod\Apple_AirPod_(lost),762D76EBD17B,40409,14453,14390
...,...,...,...,...,...,...,...
4006,4006,Samsung SmartThings Tracker (nearby),BLE Tracker\Samsung SmartThings\Samsung SmartT...,854E00A9A667,1135553,1,0
4007,4007,Tile Tracker (lost),BLE Tracker\Tile\Tile Mate\Tile_Mate_(lost),DB7E1B05AF8E,1135554,3320,3257
4008,4008,Tile Tracker (lost),BLE Tracker\Tile\Tile Mate\Tile_Mate_(nearby),DB7E1B05AF8E,1138874,3294,3231
4009,4009,Tile Tracker (lost),BLE Tracker\Tile\Tile Slim\Tile_Slim_(lost),DC90654758FA,1142168,6390,6327


In [7]:
dataset['Label'].unique()

<ArrowStringArray>
[                        'Apple Device',
       'Apple Find My Device (offline)',
        'Apple Find My Device (online)',
         'Apple Find My Tracker (lost)',
       'Apple Find My Tracker (nearby)',
     'Apple Find My Tracker (unpaired)',
   'Apple Find My Tracker Gen 2 (lost)',
 'Apple Find My Tracker Gen 2 (nearby)',
                          'DULT (lost)',
                        'DULT (nearby)',
        'Google Find My Tracker (lost)',
    'Google Find My Tracker (unpaired)',
                         'Other Device',
   'Samsung SmartThings Tracker (lost)',
 'Samsung SmartThings Tracker (nearby)',
                  'Tile Tracker (lost)']
Length: 16, dtype: string